# 🧬 Dark Manifold v5 — All Issues Fixed

## Fixes from v4

| Issue | v4 (Broken) | v5 (Fixed) |
|-------|-------------|------------|
| Km learning | `Km.mean()` - all identical | Per-reaction Km with proper gradients |
| Tau scaling | Single global tau | Per-metabolite tau |
| Substrate handling | `met_conc.mean()` proxy | Actual substrates from S matrix |
| Stoichiometry | All ±1 | Parse actual coefficients |
| Validation | None | Gene knockout + metrics |

## Architecture

```
gene_expr → gene_encoder → Vmax (per-reaction)
met_conc → substrate_conc (from S matrix) → MM kinetics
flux = Vmax * (S_conc / (Km + S_conc))  ← Per-reaction Km!
dM/dt = S @ flux
M_next = M + dt * dM/dt * tau  ← Per-metabolite tau!
```

In [ ]:
#@title 1️⃣ Setup & Upload Data
from google.colab import files
import pickle
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import re
from tqdm import tqdm

print("Upload syn3a_real_data.pkl:")
uploaded = files.upload()

with open('syn3a_real_data.pkl', 'rb') as f:
    raw_data = pickle.load(f)

print("✅ Data loaded!")

In [ ]:
#@title 2️⃣ Parse Data with PROPER Stoichiometry Coefficients

dfs = raw_data['all_dataframes']
rxns_df = pd.DataFrame(dfs['reconstruction.xlsx:Reactions'])
mets_df = pd.DataFrame(dfs['reconstruction.xlsx:Metabolites'])

metabolites = mets_df['Metabolite ID'].tolist()
met_names = mets_df['Metabolite name'].tolist()
met_to_idx = {m: i for i, m in enumerate(metabolites)}
name_to_idx = {n.lower(): i for i, n in enumerate(met_names)}

# Parse genes from GPR rules
genes = set()
gene_to_rxns = {}
gpr_rules = rxns_df['GPR rule'].tolist()

for j, gpr in enumerate(gpr_rules):
    if pd.isna(gpr):
        continue
    gene_ids = re.findall(r'MMSYN1_\d+', str(gpr))
    for g in gene_ids:
        genes.add(g)
        if g not in gene_to_rxns:
            gene_to_rxns[g] = []
        gene_to_rxns[g].append(j)

genes = sorted(list(genes))
gene_to_idx = {g: i for i, g in enumerate(genes)}

n_genes = len(genes)
n_mets = len(metabolites)
n_rxns = len(rxns_df)

def parse_stoichiometry_with_coefficients(equation):
    """
    Parse reaction equation to extract metabolites with coefficients.
    
    Examples:
        '2 H+ + ATP --> ADP + Pi' -> {H+: -2, ATP: -1, ADP: +1, Pi: +1}
        'A <==> B' -> {A: -1, B: +1} (reversible, treat as forward)
    """
    result = {}
    
    # Remove compartment markers
    eq = equation.replace('[c]:', '').replace('[e]:', '').replace('[c]', '').replace('[e]', '').strip()
    
    # Split into left and right
    if '-->' in eq:
        left, right = eq.split('-->')
    elif '<==>' in eq:
        left, right = eq.split('<==>')
    else:
        return result
    
    def parse_side(side, sign):
        parts = [p.strip() for p in side.split('+')]
        for part in parts:
            if not part:
                continue
            # Check for coefficient like "2 ATP" or "(2) ATP"
            match = re.match(r'^\(?([\d.]+)\)?\s+(.+)$', part)
            if match:
                coeff = float(match.group(1))
                met_name = match.group(2).strip()
            else:
                coeff = 1.0
                met_name = part.strip()
            
            result[met_name.lower()] = sign * coeff
    
    parse_side(left, -1)   # Substrates: negative
    parse_side(right, +1)  # Products: positive
    
    return result

# Build stoichiometry matrix with PROPER coefficients
S = np.zeros((n_mets, n_rxns))
for j, row in rxns_df.iterrows():
    equation = row['Reaction equation']
    stoich = parse_stoichiometry_with_coefficients(equation)
    
    for met_name_lower, coeff in stoich.items():
        # Find matching metabolite
        for i, name in enumerate(met_names):
            if name.lower() == met_name_lower:
                S[i, j] = coeff
                break

# Build gene-reaction matrix
G = np.zeros((n_genes, n_rxns))
for g, rxn_indices in gene_to_rxns.items():
    g_idx = gene_to_idx[g]
    for r_idx in rxn_indices:
        G[g_idx, r_idx] = 1

# Verify
connected = np.sum(np.any(S != 0, axis=1))
non_unit = np.sum(np.abs(S[S != 0]) != 1)

print(f"Genes: {n_genes}, Metabolites: {n_mets}, Reactions: {n_rxns}")
print(f"Stoichiometry non-zeros: {np.count_nonzero(S)}")
print(f"Connected metabolites: {connected}/{n_mets}", '✅' if connected == n_mets else '❌')
print(f"Non-unit coefficients: {non_unit} (coefficients like 2, 3, etc.)")

In [ ]:
#@title 3️⃣ Enhanced Loss Function

class EnhancedDeltaLoss(nn.Module):
    """Enhanced loss with Km diversity regularization."""
    
    def __init__(self, delta_weight=5.0, direction_weight=2.0, km_diversity_weight=0.1):
        super().__init__()
        self.delta_weight = delta_weight
        self.direction_weight = direction_weight
        self.km_diversity_weight = km_diversity_weight
    
    def forward(self, pred_next, true_next, true_curr, model=None):
        # State loss
        state_loss = F.mse_loss(pred_next, true_next)
        
        # Delta loss
        pred_delta = pred_next - true_curr
        true_delta = true_next - true_curr
        delta_loss = F.mse_loss(pred_delta, true_delta)
        
        # Direction loss (soft sign)
        pred_sign = torch.tanh(pred_delta * 10)
        true_sign = torch.tanh(true_delta * 10)
        direction_loss = F.mse_loss(pred_sign, true_sign)
        
        total = state_loss + self.delta_weight * delta_loss + self.direction_weight * direction_loss
        
        # Km diversity regularization - encourage different Km values
        if model is not None and hasattr(model, 'Km'):
            km_std = model.Km.std()
            # Penalize if Km values are too similar (std too low)
            km_diversity_loss = 1.0 / (km_std + 0.1)  # Lower std = higher penalty
            total = total + self.km_diversity_weight * km_diversity_loss
        
        return total, {
            'state': state_loss.item(),
            'delta': delta_loss.item(),
            'direction': direction_loss.item(),
        }

print("✅ EnhancedDeltaLoss defined")

In [ ]:
#@title 4️⃣ DarkManifoldV5 - ALL ISSUES FIXED

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

class DarkManifoldV5(nn.Module):
    """
    Dark Manifold v5 - All issues fixed.
    
    Fixes:
    1. Per-reaction Km (not Km.mean())
    2. Per-metabolite tau
    3. Actual substrate concentrations (not met_conc.mean())
    4. Proper stoichiometry coefficients
    5. Gene knockout capability
    """
    
    def __init__(self, n_genes, n_metabolites, n_reactions, S, G, hidden_dim=256):
        super().__init__()
        
        self.n_genes = n_genes
        self.n_metabolites = n_metabolites
        self.n_reactions = n_reactions
        
        self.register_buffer('S', torch.tensor(S, dtype=torch.float32))
        self.register_buffer('G', torch.tensor(G, dtype=torch.float32))
        
        # Precompute substrate mask: which metabolites are substrates for each reaction
        # Substrates have negative coefficients in S
        substrate_mask = (self.S < 0).float()  # (n_mets, n_rxns)
        self.register_buffer('substrate_mask', substrate_mask)
        
        # Gene encoder → enzyme activities (Vmax)
        self.gene_encoder = nn.Sequential(
            nn.Linear(n_genes, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, n_reactions),
            nn.Softplus(),  # Vmax > 0
        )
        
        # Metabolite encoder
        self.met_encoder = nn.Sequential(
            nn.Linear(n_metabolites, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        
        # Flux predictor
        self.flux_predictor = nn.Sequential(
            nn.Linear(hidden_dim + n_reactions, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, n_reactions),
        )
        
        # FIX #1: Per-reaction Km with diverse initialization
        # Initialize with log-uniform distribution for diversity
        init_log_km = torch.randn(n_reactions) * 0.5  # Spread around 1.0
        self.log_Km = nn.Parameter(init_log_km)
        
        # FIX #2: Per-metabolite tau (time scale)
        init_log_tau = torch.zeros(n_metabolites)
        self.log_tau = nn.Parameter(init_log_tau)
        
        # Gene regulation
        self.W_reg = nn.Parameter(torch.randn(n_genes, n_genes) * 0.01)
    
    @property
    def Km(self):
        return torch.exp(self.log_Km).clamp(0.01, 100.0)
    
    @property
    def tau(self):
        return torch.exp(self.log_tau).clamp(0.01, 10.0)
    
    def compute_substrate_concentration(self, met_conc):
        """
        FIX #3: Compute actual substrate concentration for each reaction.
        
        For each reaction, sum the concentrations of its substrates.
        This replaces the naive met_conc.mean() approach.
        
        Args:
            met_conc: (B, n_mets) metabolite concentrations
            
        Returns:
            substrate_conc: (B, n_rxns) substrate concentration per reaction
        """
        # substrate_mask: (n_mets, n_rxns), 1 where S < 0
        # met_conc: (B, n_mets)
        # Result: (B, n_rxns)
        substrate_conc = met_conc @ self.substrate_mask  # Sum of substrate concentrations
        
        # Normalize by number of substrates per reaction
        n_substrates = self.substrate_mask.sum(dim=0).clamp(min=1)  # (n_rxns,)
        substrate_conc = substrate_conc / n_substrates
        
        return substrate_conc + 0.01  # Small offset to avoid division by zero
    
    def forward(self, gene_expr, met_conc, dt=0.01):
        """
        Forward pass with all fixes applied.
        """
        # Gene regulation
        reg = torch.tanh(gene_expr @ self.W_reg.T) * 0.1
        regulated = gene_expr + reg
        
        # Gene → Vmax (enzyme activity)
        Vmax = self.gene_encoder(regulated)  # (B, n_rxns)
        
        # Encode metabolites
        met_hidden = self.met_encoder(met_conc)  # (B, hidden)
        
        # Predict flux direction/magnitude
        flux_input = torch.cat([met_hidden, Vmax], dim=-1)
        flux_raw = self.flux_predictor(flux_input)  # (B, n_rxns)
        
        # FIX #3: Actual substrate concentrations
        S_conc = self.compute_substrate_concentration(met_conc)  # (B, n_rxns)
        
        # FIX #1: Per-reaction Michaelis-Menten kinetics
        # v = Vmax * S / (Km + S)  where Km and S are per-reaction
        mm_factor = S_conc / (self.Km + S_conc)  # (B, n_rxns)
        
        flux = Vmax * flux_raw * mm_factor  # (B, n_rxns)
        
        # dM/dt = S @ flux (stoichiometry with proper coefficients)
        dM_dt = flux @ self.S.T  # (B, n_mets)
        
        # FIX #2: Per-metabolite time scale
        dM_dt = dM_dt * self.tau  # (B, n_mets) * (n_mets,) = (B, n_mets)
        
        # Euler step
        next_met = met_conc + dt * dM_dt
        next_met = next_met.clamp(min=0)  # Concentrations ≥ 0
        
        return {
            'next_metabolites': next_met,
            'flux': flux,
            'dM_dt': dM_dt,
            'Vmax': Vmax,
            'substrate_conc': S_conc,
        }
    
    def knockout(self, gene_expr, gene_name_or_idx):
        """
        Zero out a gene's expression (knockout).
        
        Args:
            gene_expr: (B, n_genes) gene expression
            gene_name_or_idx: gene name string or index
            
        Returns:
            gene_expr with the specified gene set to 0
        """
        ko_expr = gene_expr.clone()
        if isinstance(gene_name_or_idx, int):
            idx = gene_name_or_idx
        else:
            # Assume it's in genes list
            idx = gene_name_or_idx  # Will be passed as index
        ko_expr[:, idx] = 0.0
        return ko_expr
    
    def simulate_knockout(self, gene_expr, met_conc, gene_idx, n_steps=100, dt=0.01):
        """
        Simulate dynamics after gene knockout.
        
        Returns trajectory and viability assessment.
        """
        ko_expr = self.knockout(gene_expr, gene_idx)
        
        traj = [met_conc]
        current = met_conc
        
        with torch.no_grad():
            for _ in range(n_steps):
                out = self.forward(ko_expr, current, dt)
                current = out['next_metabolites']
                traj.append(current)
        
        traj = torch.stack(traj, dim=1)  # (B, T, n_mets)
        
        # Assess viability: check if key metabolites collapse
        final = traj[:, -1, :]  # (B, n_mets)
        initial = traj[:, 0, :]
        
        # Viability score: how much metabolite pool is maintained
        viability = (final.sum(dim=-1) / (initial.sum(dim=-1) + 1e-6)).mean()
        
        return {
            'trajectory': traj,
            'viability': viability.item(),
            'is_lethal': viability.item() < 0.5,  # <50% = lethal
        }


model = DarkManifoldV5(
    n_genes=n_genes,
    n_metabolites=n_mets,
    n_reactions=n_rxns,
    S=S,
    G=G,
    hidden_dim=256,
).to(device)

print(f"\nDarkManifoldV5 (All Fixes):")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Km shape: {model.log_Km.shape} (per-reaction ✅)")
print(f"  Tau shape: {model.log_tau.shape} (per-metabolite ✅)")
print(f"  Km initial std: {model.Km.std().item():.4f} (should be >0)")

In [ ]:
#@title 5️⃣ Improved Trajectory Generator

class ImprovedTrajectoryGenerator:
    """
    Trajectory generator that matches model architecture.
    
    Uses actual substrate concentrations and proper MM kinetics.
    """
    def __init__(self, S, G, n_genes, n_mets, n_rxns):
        self.S = torch.tensor(S, dtype=torch.float32)
        self.G = torch.tensor(G, dtype=torch.float32)
        self.n_genes = n_genes
        self.n_mets = n_mets
        self.n_rxns = n_rxns
        
        # Substrate mask
        self.substrate_mask = (self.S < 0).float()
        self.n_substrates = self.substrate_mask.sum(dim=0).clamp(min=1)
        
        # Random Km values for generator (will differ from model's learned Km)
        self.Km = torch.exp(torch.randn(n_rxns) * 0.5).clamp(0.1, 10.0)
        
        # Per-metabolite time scales (random for diversity)
        self.tau = torch.exp(torch.randn(n_mets) * 0.3).clamp(0.5, 2.0)
    
    def simulate(self, n_steps=50, dt=0.01, batch_size=1):
        # Random initial conditions
        gene_expr = torch.rand(batch_size, self.n_genes) * 2.0 + 0.5
        met_conc = torch.rand(batch_size, self.n_mets) * 2.0 + 0.1
        
        gene_traj = [gene_expr.clone()]
        met_traj = [met_conc.clone()]
        
        for _ in range(n_steps):
            # Enzyme activity from gene expression
            enzyme = F.relu(gene_expr) @ self.G  # (B, n_rxns)
            
            # Compute actual substrate concentrations
            substrate_conc = met_conc @ self.substrate_mask / self.n_substrates + 0.01
            
            # MM kinetics with per-reaction Km
            mm_factor = substrate_conc / (self.Km + substrate_conc)
            
            flux = enzyme * mm_factor
            
            # dM/dt = S @ flux
            dM = flux @ self.S.T
            
            # Per-metabolite tau + small noise
            dM = dM * self.tau + 0.001 * torch.randn_like(met_conc)
            
            # Update
            met_conc = (met_conc + dt * dM).clamp(min=0.001)
            gene_expr = (gene_expr + 0.002 * torch.randn_like(gene_expr)).clamp(0.1, 3.0)
            
            gene_traj.append(gene_expr.clone())
            met_traj.append(met_conc.clone())
        
        return {
            'gene_trajectory': torch.stack(gene_traj, dim=1),
            'met_trajectory': torch.stack(met_traj, dim=1),
        }

generator = ImprovedTrajectoryGenerator(S, G, n_genes, n_mets, n_rxns)

# Test
test = generator.simulate(n_steps=20, batch_size=2)
delta = (test['met_trajectory'][:, -1] - test['met_trajectory'][:, 0]).abs().mean()
print(f"✅ Generator ready | Avg change over 20 steps: {delta:.4f}")

In [ ]:
#@title 6️⃣ Train with Enhanced Loss

n_epochs = 1000
batch_size = 64
n_steps = 50
lr = 1e-3

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
loss_fn = EnhancedDeltaLoss(delta_weight=5.0, direction_weight=2.0, km_diversity_weight=0.1)

def train_step(batch):
    gene_traj = batch['gene_trajectory'].to(device)
    met_traj = batch['met_trajectory'].to(device)
    
    B, T, _ = gene_traj.shape
    total_loss = 0
    metrics = {'state': 0, 'delta': 0, 'direction': 0}
    
    for t in range(T - 1):
        out = model(gene_traj[:, t], met_traj[:, t])
        
        loss, m = loss_fn(
            out['next_metabolites'],
            met_traj[:, t + 1],
            met_traj[:, t],
            model=model,  # Pass model for Km diversity
        )
        total_loss += loss
        for k in metrics:
            metrics[k] += m[k]
    
    n = T - 1
    for k in metrics:
        metrics[k] /= n
    
    return total_loss / n, metrics

losses = []
km_stds = []  # Track Km diversity

print(f"Training v5: {n_epochs} epochs, batch={batch_size}")
print(f"Fixes: per-reaction Km, per-metabolite tau, proper substrates")

for epoch in tqdm(range(n_epochs)):
    batch = generator.simulate(n_steps=n_steps, batch_size=batch_size)
    
    optimizer.zero_grad()
    loss, metrics = train_step(batch)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()
    
    losses.append(loss.item())
    km_stds.append(model.Km.std().item())
    
    if (epoch + 1) % 200 == 0:
        print(f"Epoch {epoch+1}: Loss={loss.item():.4f} | "
              f"Km std={model.Km.std().item():.4f} | "
              f"tau std={model.tau.std().item():.4f}")

print(f"\nFinal loss: {losses[-1]:.6f}")
print(f"Final Km std: {model.Km.std().item():.4f} (should be >0, was 0 in v4)")
print(f"Final tau std: {model.tau.std().item():.4f}")

In [ ]:
#@title 7️⃣ Evaluate
import matplotlib.pyplot as plt

model.eval()

test = generator.simulate(n_steps=50, batch_size=1)
gene_test = test['gene_trajectory'].to(device)
met_test = test['met_trajectory'].to(device)

# Predict
pred_traj = [met_test[:, 0]]
current = met_test[:, 0]

with torch.no_grad():
    for t in range(50):
        out = model(gene_test[:, t], current)
        current = out['next_metabolites']
        pred_traj.append(current)

pred_traj = torch.stack(pred_traj, dim=1)

true = met_test[0].cpu().numpy()
pred = pred_traj[0].cpu().numpy()

# Metrics
corr = np.corrcoef(true.flatten(), pred.flatten())[0, 1]

true_delta = true[1:] - true[:-1]
pred_delta = pred[1:] - pred[:-1]
delta_corr = np.corrcoef(true_delta.flatten(), pred_delta.flatten())[0, 1]

# Direction accuracy
true_dir = np.sign(true[-1] - true[0])
pred_dir = np.sign(pred[-1] - pred[0])
dir_accuracy = (true_dir == pred_dir).mean()

print(f"State Correlation: {corr:.4f}")
print(f"Delta Correlation: {delta_corr:.4f}  ← KEY METRIC")
print(f"Direction Accuracy: {dir_accuracy:.1%}")

# Plot
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for i, ax in enumerate(axes.flatten()):
    ax.plot(true[:, i], 'b-', label='True', alpha=0.7, linewidth=2)
    ax.plot(pred[:, i], 'r--', label='Predicted', alpha=0.7, linewidth=2)
    ax.set_title(f'Metabolite {i}')
    ax.legend()

plt.suptitle(f'v5 Fixed | Delta Corr: {delta_corr:.3f} | Dir Acc: {dir_accuracy:.1%}')
plt.tight_layout()
plt.savefig('trajectory_v5.png', dpi=150)
plt.show()

# Km and tau distributions
fig, axes = plt.subplots(1, 2, figsize=(10, 3))

axes[0].hist(model.Km.detach().cpu().numpy(), bins=30, alpha=0.7)
axes[0].set_title(f'Km Distribution (std={model.Km.std().item():.3f})')
axes[0].set_xlabel('Km value')

axes[1].hist(model.tau.detach().cpu().numpy(), bins=30, alpha=0.7, color='orange')
axes[1].set_title(f'Tau Distribution (std={model.tau.std().item():.3f})')
axes[1].set_xlabel('Tau value')

plt.tight_layout()
plt.savefig('params_v5.png', dpi=150)
plt.show()

In [ ]:
#@title 8️⃣ Gene Knockout Validation

print("="*60)
print("GENE KNOCKOUT VALIDATION")
print("="*60)

# Generate baseline state
baseline = generator.simulate(n_steps=100, batch_size=1)
gene_baseline = baseline['gene_trajectory'][:, 0].to(device)  # Initial gene expression
met_baseline = baseline['met_trajectory'][:, 0].to(device)    # Initial metabolites

# Run wildtype simulation
wt_result = model.simulate_knockout(gene_baseline, met_baseline, gene_idx=0, n_steps=100)
wt_viability = (wt_result['trajectory'][:, -1].sum() / wt_result['trajectory'][:, 0].sum()).item()
print(f"\nWildtype viability (no knockout): {wt_viability:.2%}")

# Test knockouts for all genes
print(f"\nTesting knockouts for {n_genes} genes...")

ko_results = []
for i in tqdm(range(n_genes)):
    result = model.simulate_knockout(gene_baseline, met_baseline, gene_idx=i, n_steps=100)
    ko_results.append({
        'gene': genes[i],
        'viability': result['viability'],
        'is_lethal': result['is_lethal'],
    })

# Summary
lethal_genes = [r['gene'] for r in ko_results if r['is_lethal']]
non_lethal_genes = [r['gene'] for r in ko_results if not r['is_lethal']]

print(f"\n" + "="*60)
print(f"KNOCKOUT RESULTS")
print(f"="*60)
print(f"Total genes tested: {n_genes}")
print(f"Predicted ESSENTIAL (lethal KO): {len(lethal_genes)} ({100*len(lethal_genes)/n_genes:.1f}%)")
print(f"Predicted NON-ESSENTIAL: {len(non_lethal_genes)} ({100*len(non_lethal_genes)/n_genes:.1f}%)")

# Show most lethal knockouts
sorted_results = sorted(ko_results, key=lambda x: x['viability'])
print(f"\nMost lethal knockouts (lowest viability):")
for r in sorted_results[:10]:
    print(f"  {r['gene']}: viability={r['viability']:.2%}")

print(f"\nLeast affected knockouts (highest viability):")
for r in sorted_results[-5:]:
    print(f"  {r['gene']}: viability={r['viability']:.2%}")

In [ ]:
#@title 9️⃣ Save Model & Results

save_dict = {
    'model_state_dict': model.state_dict(),
    'genes': genes,
    'metabolites': metabolites,
    'stoichiometry': S,
    'gene_reaction_map': G,
    'training_losses': losses,
    'km_stds': km_stds,
    'knockout_results': ko_results,
    'version': 'v5_all_fixes',
    'fixes': [
        'per_reaction_km',
        'per_metabolite_tau',
        'actual_substrate_concentrations',
        'stoichiometry_coefficients',
        'gene_knockout_validation',
    ],
    'metrics': {
        'delta_corr': delta_corr,
        'state_corr': corr,
        'direction_accuracy': dir_accuracy,
        'km_final_std': model.Km.std().item(),
        'tau_final_std': model.tau.std().item(),
        'n_essential_genes': len(lethal_genes),
        'n_non_essential_genes': len(non_lethal_genes),
    }
}

torch.save(save_dict, 'dark_manifold_v5_fixed.pt')

# Summary
print("="*60)
print("DARK MANIFOLD v5 - FINAL SUMMARY")
print("="*60)
print(f"\nModel: {n_genes} genes, {n_mets} metabolites, {n_rxns} reactions")
print(f"\nAccuracy:")
print(f"  Delta Correlation: {delta_corr:.4f}")
print(f"  State Correlation: {corr:.4f}")
print(f"  Direction Accuracy: {dir_accuracy:.1%}")
print(f"\nLearned Parameters:")
print(f"  Km std: {model.Km.std().item():.4f} (v4 had 0.0!)")
print(f"  Tau std: {model.tau.std().item():.4f}")
print(f"\nGene Knockout:")
print(f"  Essential: {len(lethal_genes)}/{n_genes}")
print(f"  Non-essential: {len(non_lethal_genes)}/{n_genes}")

from google.colab import files
files.download('dark_manifold_v5_fixed.pt')
files.download('trajectory_v5.png')
files.download('params_v5.png')